In [7]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

# Setup

In [8]:
# reused from 02_baselines.ipynb
import numpy as np
import pandas as pd
from src.data import load_train_data

data = load_train_data()

TARGET = 'SalePrice'

y = np.log1p(data[TARGET])
X = data.drop(columns=[TARGET]) 



In [9]:
# feature grouping, reused from baselines
nominal_cols = ['MSZoning','MSSubClass', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType','Foundation', 'Heating','CentralAir', 'Electrical','Functional',
       'GarageType', 'GarageFinish','PavedDrive','MiscFeature',
       'SaleType', 'SaleCondition','Fence'
]

ordinal_cols = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'HeatingQC','KitchenQual','FireplaceQu','GarageQual',
                'GarageCond','PoolQC'

]
num_structured_missing = ['MasVnrArea','GarageYrBlt']
num_unstruc_missing = ['LotFrontage']

complete_num_cols = [
  col for col in data.columns
  if col not in nominal_cols + ordinal_cols + ["SalePrice"] + num_structured_missing + num_unstruc_missing
]

# check if I got everything
print(len(nominal_cols)+len(ordinal_cols)+len(num_structured_missing)+len(num_unstruc_missing)+len(complete_num_cols) +1)#1 for the saleprice

81


# Feature Engineering

I will use the feature matrix X to conduct feature engineering, data is kept isolated 

It was observed that there are a lot of premium amenities a house can have, such as pool, garage, Fireplace, masonry veneer ...  And certainly not all the houses have them, however, in the dataset, they are all represented by a structured missingness in some attributes, NaN means no such amenity, It will perhaps be beneficial if a binary feature can be engineered to indicate if the house has a certain feature

Previously in EDA, we identified some attributes with significant percentage of missing values, they are actually the rare amentities in the house, I have listed them below:
1. Pool
2. Miscelaneous feature not covered in other categories
3. Alley
4. Fence
5. Masonry Veneer
6. Fireplace

I will start with these six rare amentities

Except from these 6 features, there are also other more common but still premium features: 
1. Garage
2. Basement 

In [10]:
# add binary indicators for amenities 
X['HasPool'] = X['PoolQC'].notna().astype(int) # 1 has pool, 2 no pool
X['HasMisc'] = X['MiscFeature'].notna().astype(int)
X['HasAlley'] = X['Alley'].notna().astype(int)
X['HasFence'] = X['Fence'].notna().astype(int)
X['HasMasVnr'] = (X['MasVnrArea']>0).astype(int)
X['HasFireplace'] = X['FireplaceQu'].notna().astype(int)
X['HasBasement'] = X['BsmtQual'].notna().astype(int)
X['HasGarage'] = X['GarageType'].notna().astype(int)